# Chapter 02 — Data Cleaning: Live Examples

**Session 1 | Chapter 2 | Duration: ~15 min**

This notebook demonstrates all data cleaning techniques with a realistic messy dataset.
We simulate a **student survey dataset** with intentional messiness.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

print('Libraries loaded!')

## 1. Creating a Messy Dataset

In real life, you get this data from a CSV, database, or API.  
Here we create it manually so we know exactly what's wrong.

In [ ]:
# Simulate a messy student dataset
raw_data = {
    'student_id': range(1, 21),
    'age': [22, 25, 23, None, 21, 999, 24, 22, 26, 23,
            20, 25, None, 22, 24, 21, 23, 25, 22, 24],
    'gender': ['Male', 'female', 'MALE', 'Female', None, 'male', 'Female', 'Male',
               'M', 'Female', 'male', 'Female', 'Male', None, 'female', 'Male',
               'Female', 'male', 'FEMALE', 'Male'],
    'study_hours_per_week': [15, 20, None, 18, 25, 12, 30, 22, None, 16,
                              19, 28, 14, 21, 17, None, 23, 26, 18, 150],  # 150 = outlier
    'grade': ['A', 'B', 'A', 'C', 'B', None, 'A', 'B', 'C', 'A',
               'B', 'A', 'C', 'B', None, 'A', 'B', 'C', 'A', 'B'],
    'faculty': ['Science', 'Arts', 'Science', 'Engineering', 'Arts',
                 'Science', None, 'Arts', 'Engineering', 'Science',
                 'Arts', 'Science', 'Engineering', 'Arts', 'Science',
                 'Engineering', 'Arts', 'Science', 'Arts', 'Engineering'],
    'score': [82, 75, 91, 68, 78, 55, 95, 80, 72, 88,
               76, 92, 65, 79, 83, 70, 84, 77, 90, 73]
}

df = pd.DataFrame(raw_data)
print('Raw dataset shape:', df.shape)
df.head(10)

## 2. Inspection — Always Start Here

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = df.isnull().mean().round(3) * 100
missing_report = pd.DataFrame({'count': missing, 'percent': missing_pct})
print(missing_report[missing_report['count'] > 0])

In [ ]:
# Visual missing-value map
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
            cmap='Blues', ax=ax)
ax.set_title('Missing Value Map (blue = missing)', fontsize=13)
ax.set_xlabel('Columns')
plt.tight_layout()
plt.show()

## 3. Fixing the `gender` Column — Inconsistent Categories

Problem: 'Male', 'male', 'MALE', 'M' all mean the same thing.

In [ ]:
print('Unique gender values before cleaning:')
print(df['gender'].unique())

# Step 1: Standardize to lowercase and strip whitespace
df['gender'] = df['gender'].str.lower().str.strip()

# Step 2: Map shorthand 'm' → 'male'
df['gender'] = df['gender'].replace({'m': 'male', 'f': 'female'})

print('\nUnique gender values after cleaning:')
print(df['gender'].unique())

## 4. Handling Missing Values

In [ ]:
# --- Numerical: fill with median (robust to outliers) ---
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
print(f'age: filled missing values with median = {age_median}')

study_median = df['study_hours_per_week'].median()
df['study_hours_per_week'] = df['study_hours_per_week'].fillna(study_median)
print(f'study_hours: filled missing values with median = {study_median}')

# --- Categorical: fill with mode (most frequent) ---
for col in ['gender', 'grade', 'faculty']:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f'{col}: filled missing values with mode = "{mode_val}"')

print('\nMissing values remaining:', df.isnull().sum().sum())

## 5. Detecting and Treating Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['age', 'study_hours_per_week', 'score']):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.6))
    ax.set_title(f'Boxplot: {col}', fontsize=12)
    ax.set_ylabel(col)

plt.suptitle('Boxplots — Outlier Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('study_hours max value:', df['study_hours_per_week'].max())
print('→ 150 hours/week is impossible (there are only 168 hours in a week!)')

In [ ]:
# IQR-based outlier detection for study_hours
Q1 = df['study_hours_per_week'].quantile(0.25)
Q3 = df['study_hours_per_week'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}')
print(f'Outlier bounds: [{lower_bound:.1f}, {upper_bound:.1f}]')

# Show outliers
outliers = df[(df['study_hours_per_week'] < lower_bound) | 
              (df['study_hours_per_week'] > upper_bound)]
print(f'\nOutliers found:\n{outliers[["student_id", "study_hours_per_week"]]}')

# Strategy: clip to upper bound (winsorization)
df['study_hours_per_week'] = df['study_hours_per_week'].clip(lower=lower_bound, upper=upper_bound)
print(f'\nAfter clipping — max value: {df["study_hours_per_week"].max()}')

# Also fix age = 999 (clearly erroneous)
df.loc[df['age'] > 100, 'age'] = age_median
print(f'age max after fix: {df["age"].max()}')

## 6. Encoding Categorical Features

ML models need numbers. Let's convert our categorical columns.

In [ ]:
# --- Binary encoding: gender ---
df['gender_encoded'] = (df['gender'] == 'female').astype(int)
print('Gender encoding:')
print(df[['gender', 'gender_encoded']].value_counts())

# --- Ordinal encoding: grade (has natural order: C < B < A) ---
grade_order = {'C': 0, 'B': 1, 'A': 2}
df['grade_encoded'] = df['grade'].map(grade_order)
print('\nGrade encoding:', grade_order)

In [ ]:
# --- One-hot encoding: faculty (nominal, no natural order) ---
faculty_dummies = pd.get_dummies(df['faculty'], prefix='faculty', dtype=int)
print('One-hot encoded faculty:')
print(faculty_dummies.head(8))

# Add to dataframe
df = pd.concat([df, faculty_dummies], axis=1)

## 7. Feature Scaling

Before scaling: notice the different scales of our numerical features.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

numerical_features = ['age', 'study_hours_per_week', 'score']

print('=== Before Scaling ===')
print(df[numerical_features].describe().round(2))

In [ ]:
# StandardScaler (Z-score): mean=0, std=1
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numerical_features] = scaler.fit_transform(df[numerical_features])

print('=== After StandardScaler ===')
print(df_scaled[numerical_features].describe().round(3))

In [ ]:
# Visual comparison: before vs after scaling
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(numerical_features):
    # Before
    axes[0, i].hist(df[col], bins=10, color='#3498db', alpha=0.7, edgecolor='white')
    axes[0, i].set_title(f'{col}\n(original)', fontsize=11)
    axes[0, i].set_ylabel('Count')

    # After
    axes[1, i].hist(df_scaled[col], bins=10, color='#e74c3c', alpha=0.7, edgecolor='white')
    axes[1, i].set_title(f'{col}\n(standardized)', fontsize=11)
    axes[1, i].set_ylabel('Count')

axes[0, 0].set_ylabel('Count (original)', fontsize=11)
axes[1, 0].set_ylabel('Count (scaled)', fontsize=11)
plt.suptitle('Feature Distributions: Before vs After Standardization', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('\nNote: the SHAPE of the distribution is unchanged.')
print('Only the scale (axis values) changed.')

## 8. Train / Test Split

The final step before any modeling: split the data.

**This is the moment to do it — after all cleaning, before any model.**

In [ ]:
from sklearn.model_selection import train_test_split

# Select the features we want to use
feature_cols = ['age', 'gender_encoded', 'study_hours_per_week',
                'grade_encoded', 'faculty_Arts', 'faculty_Engineering', 'faculty_Science']
target_col = 'score'

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Training set:  {X_train.shape[0]} samples')
print(f'Test set:      {X_test.shape[0]} samples')
print(f'Features used: {X_train.shape[1]}')
print('\nThe test set is now locked away — we will only use it to evaluate the final model.')

## Summary

We went from a **messy raw dataset** to **model-ready data** in 7 steps:

| Step | What we did |
|------|------------|
| 1. Inspect | Shape, dtypes, missing values |
| 2. Fix inconsistencies | Standardized 'gender' values |
| 3. Handle missing values | Median for numerical, mode for categorical |
| 4. Remove/fix outliers | Clipped study hours, fixed age=999 |
| 5. Encode categories | Binary, ordinal, and one-hot encoding |
| 6. Scale features | StandardScaler on numerical columns |
| 7. Split data | 80% train, 20% test |

→ Now open the **exercises** and try it yourself!